# Exercício 9 — Massa corporal e medidas do bico de pinguins

**Enunciado:** O conjunto `penguins` (projeto `palmerpenguins`) reúne medidas de pinguins de três espécies observados no arquipélago Palmer, na Antártida, entre 2007 e 2009. As variáveis `species`, `sex` e `body_mass_g` indicam espécie, sexo e massa corporal em gramas; `bill_length_mm` e `bill_depth_mm` registram o comprimento e a altura do bico, em milímetros.

Considere apenas os pinguins da espécie **Chinstrap**. Em cada análise, use os indivíduos com valores conhecidos nas variáveis necessárias e suponha que as observações de indivíduos distintos sejam independentes.

- (a) Separe por `sex` (`male`, `female`). Informe o número de indivíduos, a média e o desvio padrão da massa corporal em cada grupo, em quilogramas. Represente as distribuições com boxplots ou histogramas.
- (b) A massa corporal média difere entre machos e fêmeas? Formule $H_0$ e $H_1$, proponha uma estatística de teste, calcule seu valor observado e o p-valor. Encontre um IC de 95% para a diferença entre as médias. Apresente a decisão para $\alpha = 0.05$ e interprete a diferença estimada em kg.
- (c) Só as fêmeas: diagrama de dispersão entre comprimento e altura do bico e coeficiente de correlação de Pearson $r$. Quantos pares foram usados? O que o sinal e a magnitude de $r$ sugerem?
- (d) A correlação do item (c) poderia surgir ao acaso se comprimento e altura fossem independentes? Teste de permutação: manter os comprimentos fixos, embaralhar as alturas, recalcular $|r|$, repetir $B = 10000$ vezes, fazer o histograma e estimar o p-valor. Por que usamos o valor absoluto? Por que o embaralhamento representa a independência? Decisão para $\alpha = 0.05$.
- (e) Redija as conclusões de (b) e (d). Se (d) não rejeitar $H_0$, isso permite afirmar independência? Como responder a quem interpreta o p-valor como a probabilidade de $H_0$ ser verdadeira?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import erf, sqrt

def Phi(x):
    # CDF da N(0,1)
    return 0.5 * (1 + erf(x / sqrt(2)))

rng = np.random.default_rng(42)

pinguins = pd.read_csv('../dados_lista3/penguins.csv')
chinstrap = pinguins[pinguins['species'] == 'Chinstrap']
chinstrap.head()

## Item (a)

In [ ]:
dados_a = chinstrap.dropna(subset=['sex', 'body_mass_g'])

machos = dados_a[dados_a['sex'] == 'male']['body_mass_g'] / 1000   # em kg
femeas = dados_a[dados_a['sex'] == 'female']['body_mass_g'] / 1000

print(f"Machos: n = {len(machos)}, media = {machos.mean():.3f} kg, dp = {machos.std():.3f} kg")
print(f"Femeas: n = {len(femeas)}, media = {femeas.mean():.3f} kg, dp = {femeas.std():.3f} kg")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))

ax[0].boxplot([machos, femeas])
ax[0].set_xticks([1, 2], ['Machos', 'Femeas'])
ax[0].set_ylabel('massa (kg)')
ax[0].set_title('Boxplots')

ax[1].hist(machos, bins=10, range=(2.5, 5))
ax[1].set_title('Machos')
ax[1].set_xlabel('massa (kg)')

ax[2].hist(femeas, bins=10, range=(2.5, 5))
ax[2].set_title('Femeas')
ax[2].set_xlabel('massa (kg)')

plt.tight_layout()
plt.show()

**Comentário:** As duas distribuições são razoavelmente simétricas. Os machos ficam visivelmente acima: mediana ≈ 3.95 kg contra ≈ 3.55 kg nas fêmeas. As fêmeas têm um pouco menos de variação. Os boxplots marcam um ponto extremo em cada grupo (um macho de 4.8 kg e uma fêmea de 2.7 kg), mas nada que pareça erro de medida.

## Item (b)

Parâmetro: $\delta = \mu_M - \mu_F$.
$$H_0: \delta = 0 \quad \text{versus} \quad H_1: \delta \neq 0.$$

$$\hat\delta = \overline{M} - \overline{F}, \qquad \widehat{\text{se}} = \sqrt{\frac{S_M^2}{n_M} + \frac{S_F^2}{n_F}} \quad \text{(plug-in)}, \qquad W = \frac{\hat\delta}{\widehat{\text{se}}} \approx N(0,1) \text{ sob } H_0.$$

Rejeitamos $H_0$ se $|W| > z_{\alpha/2}$; p-valor $= 2\Phi(-|w|)$; IC de 95%: $\hat\delta \pm 1.96\,\widehat{\text{se}}$.

In [ ]:
delta_hat = machos.mean() - femeas.mean()
se_hat = np.sqrt(machos.var() / len(machos) + femeas.var() / len(femeas))

W = delta_hat / se_hat
p_valor = 2 * Phi(-abs(W))

print(f"delta_hat = {delta_hat:.3f} kg")
print(f"se_hat    = {se_hat:.3f} kg")
print(f"W         = {W:.3f}")
print(f"p-valor   = {p_valor:.2e}")
print(f"IC 95%    = ({delta_hat - 1.96*se_hat:.3f}, {delta_hat + 1.96*se_hat:.3f}) kg")
print("alpha = 0.05:", "rejeita H0" if p_valor < 0.05 else "nao rejeita H0")

**Interpretação:** Estimo que os machos Chinstrap pesam, em média, cerca de **0.41 kg a mais** que as fêmeas (3.94 kg contra 3.53 kg). Como $|w| \approx 5.21$ e o p-valor é $\approx 2 \times 10^{-7} < 0.05$, rejeito $H_0$: a massa média difere entre os sexos. O IC de 95%, $(0.26,\ 0.57)$ kg, não contém zero e diz que a diferença plausível fica entre uns 260 g e 570 g. A estimativa pontual equivale a cerca de 12% da massa média de uma fêmea.

## Item (c)

Só as fêmeas, com comprimento e altura do bico conhecidos.

In [ ]:
dados_c = chinstrap[chinstrap['sex'] == 'female'].dropna(subset=['bill_length_mm', 'bill_depth_mm'])

comprimento = dados_c['bill_length_mm'].to_numpy()
altura = dados_c['bill_depth_mm'].to_numpy()

r_obs = np.corrcoef(comprimento, altura)[0, 1]

print(f"numero de pares = {len(comprimento)}")
print(f"r = {r_obs:.3f}")

In [ ]:
plt.scatter(comprimento, altura)
plt.xlabel('comprimento do bico (mm)')
plt.ylabel('altura do bico (mm)')
plt.title(f'Femeas Chinstrap (r = {r_obs:.2f})')
plt.show()

**Interpretação:** Usei os 34 pares de fêmeas (nenhum valor faltante). O $r \approx 0.26$ é **positivo**: fêmeas com bico mais comprido tendem a ter bico um pouco mais alto. Mas a magnitude é **fraca**: no gráfico a nuvem de pontos é bem espalhada e não fica perto de uma reta. Com só 34 pares, uma correlação desse tamanho pode ter surgido por acaso, e o item (d) testa isso.

## Item (d) — teste de permutação

$H_0$: comprimento e altura do bico são independentes.

Se forem independentes, o pareamento original não importa: qualquer altura poderia estar ao lado de qualquer comprimento. Então, embaralhando as alturas (sem reposição), geramos dados que **poderiam ter sido observados sob $H_0$**, e o $|r|$ desses dados mostra o tamanho de correlação que aparece só por acaso.

Passos:
1. manter `comprimento` fixo e embaralhar `altura`;
2. calcular $|r|$ dos novos pares;
3. repetir $B = 10000$ vezes;
4. p-valor = fração das repetições com $|r| \geq |r_{obs}|$.

In [ ]:
B = 10_000
r_perm = np.zeros(B)

for b in range(B):
    altura_embaralhada = rng.permutation(altura)
    r_perm[b] = abs(np.corrcoef(comprimento, altura_embaralhada)[0, 1])

p_valor_perm = np.mean(r_perm >= abs(r_obs))

print(f"|r_obs|  = {abs(r_obs):.3f}")
print(f"p-valor  = {p_valor_perm:.4f}")
print("alpha = 0.05:", "rejeita H0" if p_valor_perm < 0.05 else "nao rejeita H0")

In [ ]:
plt.hist(r_perm, bins=50)
plt.axvline(abs(r_obs), color='r', linestyle='--', label=f'|r_obs| = {abs(r_obs):.2f}')
plt.xlabel('|r| sob embaralhamento')
plt.title('Distribuicao de |r| sob H0 (independencia)')
plt.legend()
plt.show()

**Discussão:**

Uso o valor absoluto porque o teste é **bilateral**: $H_1$ é "existe associação", e uma correlação forte negativa seria tão incompatível com a independência quanto uma positiva. Com $|r|$, valores longe de zero nos dois sentidos contam como extremos.

O embaralhamento representa a independência porque, se comprimento e altura forem independentes, saber o comprimento de um pinguim não diz nada sobre a altura do bico. Então qualquer altura poderia estar pareada com qualquer comprimento, e todas as permutações são igualmente prováveis. Embaralhar as alturas destrói qualquer associação, mas mantém a distribuição de cada variável, e com isso gera dados que poderiam ter sido observados sob $H_0$.

No histograma, $|r_{obs}| \approx 0.26$ fica no meio da distribuição de $|r|$ sob $H_0$: cerca de 14% das permutações deram $|r|$ maior ou igual. Com p-valor $\approx 0.14 > 0.05$, **não rejeito $H_0$**. Os dados não trazem evidência suficiente de associação entre comprimento e altura do bico nas fêmeas.

## Item (e)

**Conclusões:**
- **(b)** Há forte evidência de que a massa média dos Chinstrap difere entre os sexos: os machos pesam em média cerca de 0.41 kg a mais que as fêmeas, com IC de 95% de 0.26 a 0.57 kg.
- **(d)** Nas fêmeas, a correlação entre comprimento e altura do bico é fraca ($r \approx 0.26$) e compatível com o acaso (p-valor de permutação $\approx 0.14$). Não encontrei evidência de associação.

**Não rejeitar $H_0$ não permite afirmar independência.** Ausência de evidência não é evidência de ausência: com só 34 fêmeas, o teste tem pouco poder para detectar uma correlação fraca, e uma associação real dessa ordem pode existir sem ser detectada. Além disso, $r$ só mede associação **linear**, e as variáveis poderiam ser dependentes de outra forma. Só posso dizer que os dados são compatíveis com a independência.

**Sobre o p-valor:** ele **não** é a probabilidade de $H_0$ ser verdadeira. O p-valor é calculado **supondo** $H_0$ verdadeira: é $P(\text{obter um resultado tão ou mais extremo que o observado} \mid H_0)$, e não $P(H_0 \mid \text{dados})$. No item (d), por exemplo, $p \approx 0.14$ quer dizer que, se as variáveis fossem independentes, cerca de 14% dos embaralhamentos dariam $|r| \geq 0.26$. Não quer dizer que há 14% de chance de elas serem independentes. Para falar em probabilidade de $H_0$ seria preciso uma abordagem bayesiana, com uma probabilidade a priori.